# D1 — What is over-squashing? / ¿Qué es el over-squashing?

🇬🇧 **The problem in one picture.** A graph neural network (GNN) updates each node by mixing in information
from its neighbours, one hop at a time. If a node needs information that lives *far away* — and the graph has
a **bottleneck** (a narrow place all the information must pass through) — then a huge amount of distant
information gets crammed into one fixed-size vector. That cramming is **over-squashing**, and it makes the
network fail at long-range tasks.

🇪🇸 **El problema en una imagen.** Una red neuronal de grafos (GNN) actualiza cada nodo mezclando información
de sus vecinos, un salto a la vez. Si un nodo necesita información que está *lejos* — y el grafo tiene un
**cuello de botella** (un lugar estrecho por el que toda la información debe pasar) — entonces una cantidad
enorme de información distante se aprieta en un único vector de tamaño fijo. Ese apretujamiento es el
**over-squashing**, y hace que la red falle en tareas de largo alcance.

<img src="img/01_bottleneck.svg" width="720" alt="A bottleneck graph: sources feed through a narrow middle into a target; many messages get squashed into one vector."/>

<sub>🇬🇧 The hourglass: every path from a source to the target must cross the narrow middle. · 🇪🇸 El reloj de arena: todo camino de una fuente al objetivo debe cruzar el medio estrecho.</sub>

## A concrete bottleneck / Un cuello de botella concreto

🇬🇧 We build a tiny graph shaped like an hourglass: a few **sources** on one side, one **target** on the
other, and a narrow **middle layer** every path must cross. The target must recover a fact about the sources.

🇪🇸 Construimos un grafo pequeño con forma de reloj de arena: unas pocas **fuentes** de un lado, un
**objetivo** del otro, y una **capa intermedia** estrecha que todo camino debe cruzar. El objetivo debe
recuperar un dato sobre las fuentes.

In [ ]:
import torch, numpy as np, networkx as nx, matplotlib.pyplot as plt
from oversquash.data_bottleneck import make_bottleneck_graph

g = torch.Generator().manual_seed(0)
K, M, depth = 4, 3, 2          # 4 sources, middle layers of 3, depth 2
data = make_bottleneck_graph(K, M, depth, g)
print(f'{data.num_nodes} nodes, target node = {int(data.root_mask.nonzero()[0])}')

In [ ]:
# Draw it / Dibujarlo
G = nx.DiGraph(); G.add_edges_from(data.edge_index.t().tolist())
t = int(data.root_mask.nonzero()[0])
# layered layout: sources left, target right
layer = {}
for n in range(K): layer[n] = (0, K/2 - n)
mid = list(range(K, data.num_nodes-1)); per = M
for idx, n in enumerate(mid): layer[n] = (1 + idx//per, per/2 - (idx%per))
layer[t] = (1 + len(mid)//per + 1, 0)
colors = ['#60a5fa']*K + ['#fbbf24']*(data.num_nodes-1-K) + ['#34d399']
plt.figure(figsize=(7,4))
nx.draw(G, layer, node_color=colors, with_labels=True, node_size=700,
        font_size=9, arrowsize=12, edge_color='#cbd5e1')
plt.title('sources (blue) → bottleneck (yellow) → target (green)')
plt.show()

## Why it gets squashed / Por qué se aprieta

🇬🇧 Count the **walks** of length `depth+1` that reach the target. Each is a distinct way information could
have flowed. A normal GNN, after that many layers, has to compress *all* of them into the target's single
vector. The count below is how many things get squashed into one.

🇪🇸 Cuenta los **recorridos (walks)** de longitud `depth+1` que llegan al objetivo. Cada uno es una forma
distinta en que la información pudo fluir. Una GNN normal, tras esas capas, tiene que comprimir *todos* ellos
en el único vector del objetivo. El número de abajo es cuántas cosas se aprietan en uno solo.

In [ ]:
from oversquash.ideal_bridge import walk_operators
ei, N = data.edge_index.numpy(), data.num_nodes
raw, _ = walk_operators(ei, N, max_length=depth+1)
squashed = int(raw[depth][:, t].sum())
print(f'walks of length {depth+1} reaching the target: {squashed}')
print(f'... all squashed into ONE vector of fixed width. That is over-squashing.')
print('\nNow make the graph deeper and watch it explode / Hazlo más profundo y mira cómo explota:')
for d in [1,2,3,4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    r,_ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    tt = int(dd.root_mask.nonzero()[0])
    print(f'  depth {d}: {int(r[d][:,tt].sum()):5d} messages squashed into one vector')

## The takeaway / La conclusión

🇬🇧 The number of messages a node must absorb grows **exponentially** with distance, but the vector that holds
them stays the same size. Something has to give — and what gives is the network's ability to tell the messages
apart. In **D2** we'll learn the precise tool for *counting and reasoning about* these paths: the path algebra
of a quiver.

🇪🇸 El número de mensajes que un nodo debe absorber crece **exponencialmente** con la distancia, pero el vector
que los guarda mantiene el mismo tamaño. Algo tiene que ceder — y lo que cede es la capacidad de la red para
distinguir los mensajes. En **D2** aprenderemos la herramienta precisa para *contar y razonar sobre* estos
caminos: el álgebra de caminos de un quiver.